# RPS-GMM: Sentinel-1 backscatter and Sentinel-2 water percentage

Classifies supraglacial lakes from the Sentinel-1 backscatter anomaly $HV_{anom}$ combined with the Sentinel-2 water percentage $p_{water}$. Each channel is embedded separately with the same $(\tau, d)$ and the delay vectors are concatenated.

This is the second row of the paper's headline comparison (Figure 4).

The model trains on **one representative lake per class** and is evaluated over
all 777 labeled lakes.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src" / "rpsgmm").is_dir():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError("Run this notebook from inside the repository.")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

print("Repository root:", REPO_ROOT)

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report

from rpsgmm import CLASSES, RPSGMMClassifier, load_backscatter_water
from rpsgmm.viz import plot_class_examples, plot_confusion_matrix, plot_phase_spaces

warnings.filterwarnings("ignore")
np.random.seed(42)

## Load the data

In [ ]:
X_train, y_train, train_ids = load_backscatter_water("representative")
X_eval, y_eval, eval_ids = load_backscatter_water("full")

print(f"train {X_train.shape}  ->  {[CLASSES[c] for c in y_train]}")
print(f"eval  {X_eval.shape}")
print()
for code, name in enumerate(CLASSES):
    print(f"  {name:9s} {int((y_eval == code).sum()):4d} lakes")

In [ ]:
fig = plot_class_examples(X_train, y_train)
plt.show()

## Grid search over $(\tau, d)$

Algorithm 1 searches the time delay $\tau \in [2, 30]$ and embedding dimension
$d \in [3, 30]$, keeping the pair with the highest accuracy.

This evaluates 812 combinations and takes several minutes. Narrow the ranges
below for a quick check.

In [ ]:
TAU_RANGE = range(2, 31)
D_RANGE = range(3, 31)

# For a fast smoke test, uncomment:
# TAU_RANGE, D_RANGE = range(2, 15, 3), range(3, 16, 3)

clf = RPSGMMClassifier(n_components=10, n_init=1, random_state=42)
grid = clf.grid_search(
    X_train, y_train, X_eval, y_eval, TAU_RANGE, D_RANGE, n_jobs=-1, verbose=False
)

print(f"\nBest: tau={grid.tau}, d={grid.d}, accuracy={grid.accuracy * 100:.2f}%")

In [ ]:
grid.to_frame().head(10)

## Fit at the selected operating point

In [ ]:
clf.fit(X_train, y_train)
predictions = clf.predict(X_eval)
accuracy = (predictions == y_eval).mean()

print(f"Accuracy: {accuracy * 100:.2f}%   (tau={clf.tau}, d={clf.d})\n")
print(classification_report(y_eval, predictions, target_names=CLASSES, digits=4))

In [ ]:
fig = plot_phase_spaces(X_train, y_train, clf.tau, clf.d)
plt.show()

In [ ]:
fig = plot_confusion_matrix(clf.confusion_matrix(X_eval, y_eval), accuracy)
plt.show()

## Per-class likelihoods

Each lake is assigned to the class whose GMM gives its phase-space trajectory
the highest mean log-likelihood (paper, Equation 8).

In [ ]:
scores = clf.log_likelihoods(X_eval[:5])
import pandas as pd
pd.DataFrame(scores, columns=[f"log p(X | {c})" for c in CLASSES],
             index=eval_ids[:5]).round(2)

---

The equivalent command-line run, which also writes `results/`:

```bash
python scripts/run_rps_gmm.py --features backscatter_water
```